# 07 - Saving, loading, and inference

**Goals:** save weights, preprocessing, feature order, and architecture; reload and verify predictions.

After selecting weights as in [06](06_regularization.ipynb), preserve everything required for inference. This standalone example trains the simpler baseline from notebook 05. Run cells in order in a fresh kernel; each notebook supplies its own setup.

## Prepare the data

Recreate the training-fitted tensors from notebook 03. Test rows stay reserved for the final assessment in notebook 08.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Keep these small CPU examples quick.


In [ ]:
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."

df = pd.read_csv(data_path)

features = ["feature_1", "feature_2", "feature_3"]
train = df[df["split"] == "train"]
validation = df[df["split"] == "validation"]
test = df[df["split"] == "test"]


In [ ]:
scaler = StandardScaler().fit(train[features])  # Fit only on training rows.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)


In [ ]:
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)


In [ ]:
model = nn.Sequential(nn.Linear(len(features), 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


## Train weights for this example

Run the familiar batch updates for a fixed 20 epochs. An inference checkpoint can store either these final weights or the validation-selected weights from notebook 06.

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    for x_batch, y_batch in loader:
        optimizer.zero_grad()  # Each batch needs fresh gradients.
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()


In [ ]:
for epoch in range(20):
    train_one_epoch(model, train_loader)
model.eval()


## Assemble an inference checkpoint

A state dictionary contains parameter tensors. Also save the architecture settings, ordered feature names, class mapping, and training-fitted means and scales. More complex preprocessing needs its own saved transformations.

In [ ]:
checkpoint = {
    "model_state": model.state_dict(),
    "n_features": len(features), "hidden_units": 16, "n_classes": 2,
    "feature_names": features, "class_names": ["class 0", "class 1"],
    "mean": torch.tensor(scaler.mean_, dtype=torch.float32),
    "scale": torch.tensor(scaler.scale_, dtype=torch.float32),
}


Write the checkpoint into the Git-ignored `artifacts` directory. Rerunning this cell replaces this example file.

In [ ]:
artifact_dir = root / "artifacts"
artifact_dir.mkdir(exist_ok=True)
checkpoint_path = artifact_dir / "classifier.pt"
torch.save(checkpoint, checkpoint_path)
print("saved:", checkpoint_path)


## Recreate the model

Load trusted files only. `weights_only=True` restricts deserialization; `map_location="cpu"` places loaded tensors on the CPU. Recreate the same layer order and dimensions before loading the weights.

In [ ]:
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = nn.Sequential(
    nn.Linear(saved["n_features"], saved["hidden_units"]),
    nn.ReLU(),
    nn.Linear(saved["hidden_units"], saved["n_classes"]),
)
restored.load_state_dict(saved["model_state"])
restored.eval()


Check the reloaded computation on the same validation inputs. This verifies the saved weights and architecture, before introducing raw-data preprocessing.

In [ ]:
with torch.no_grad():
    torch.testing.assert_close(model(x_val), restored(x_val))
print("Reloaded model matches the original on validation inputs.")


## Preprocess new raw observations

Select columns in the saved feature order and reuse the saved training means and scales. Do not fit a new scaler on incoming observations.

In [ ]:
new_rows = pd.DataFrame(
    [[0.5, -1.0, 0.2], [-1.0, 1.5, 0.0]], columns=saved["feature_names"]
)
raw = torch.tensor(new_rows[saved["feature_names"]].to_numpy(), dtype=torch.float32)
standardized = (raw - saved["mean"]) / saved["scale"]
print("new input shape:", standardized.shape)


Verify the saved preprocessing too: it should match the original fitted scaler on these raw rows, within floating-point precision.

In [ ]:
expected = torch.tensor(scaler.transform(new_rows[features]), dtype=torch.float32)
torch.testing.assert_close(standardized, expected)
print("Saved preprocessing matches the original scaler.")


## Predict and interpret

Use evaluation mode and disable gradient recording. Softmax produces one probability per class; map the largest score back to its saved class name.

In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(restored(standardized), dim=1)
    predictions = probabilities.argmax(dim=1)

print("probabilities:", probabilities)
print("predicted labels:", [saved["class_names"][i] for i in predictions.tolist()])


To resume training, also save optimizer state, epoch, and any scheduler state. Exact continuation can additionally require random-generator and sampler state; this inference checkpoint alone does not resume Adam exactly.

## Try it

1. Change a raw row and interpret the predicted class and probabilities.

2. Compare predictions with and without standardization. Why does saving the scaler matter?

3. Save and reload `optimizer.state_dict()` with an epoch number for a resumable training checkpoint.

4. Add a class-label mapping for a dataset whose labels are strings.

**Next:** [Your own model](08_your_own_model.ipynb) combines the workflow into a baseline and a controlled experiment.